# Lab | M-Shots FAQ Chatbot

This notebook builds a small chatbot that answers questions using a short FAQ and the OpenAI API. It keeps the conversation history so it can understand follow-up questions.

## Setup

Set your API key in the environment **before starting Jupyter**. In PowerShell, run `$env:OPENAI_API_KEY="your-api-key"` and then start Jupyter from that same PowerShell window. Never paste your real key into a notebook cell or save it in this file. The optional `OPENAI_MODEL` environment variable changes the model; the default is `gpt-4o-mini`.

Run the cells in order. The final cell starts the chat; type `quit` or `exit` to stop.

In [1]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

## FAQ knowledge base

Edit or add entries here to change what the chatbot knows about the lab.

In [2]:
FAQS = [
    {
        "question": "How do I complete the lab?",
        "answer": (
            "Follow the instructions in the Jupyter notebook, answer each "
            "exercise, and test your answers in the cells."
        ),
    },
    {
        "question": "How do I save my work?",
        "answer": (
            "Jupyter Notebook saves progress automatically, but you should "
            "also save your work manually from time to time."
        ),
    },
    {
        "question": "What do I need to submit?",
        "answer": "Submit the downloaded notebook containing your exercise responses.",
    },
    {
        "question": "How do I submit the completed lab?",
        "answer": (
            "Add the deliverable to git, commit and push your branch, create "
            "a pull request, then paste its link into the Student Portal."
        ),
    },
]

faq_text = "\n".join(
    f"Q: {item['question']}\nA: {item['answer']}" for item in FAQS
)

SYSTEM_PROMPT = f"""You are a helpful assistant for the Lab | M-Shots exercise.
Answer questions using the FAQ below. You may use the conversation to
understand follow-up questions, but do not invent lab policies or facts that
are not in the FAQ. If the FAQ does not contain the answer, say so clearly.

FAQ:
{faq_text}
"""

print(f"Loaded {len(FAQS)} FAQ entries.")

Loaded 4 FAQ entries.


## OpenAI API helper

This function sends the complete message history to the Chat Completions API and returns the assistant's reply. API and network errors are reported clearly.

In [3]:
def ask_openai(messages: list[dict[str, str]], api_key: str, model: str) -> str:
    request = Request(
        "https://api.openai.com/v1/chat/completions",
        data=json.dumps({"model": model, "messages": messages}).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urlopen(request, timeout=60) as response:
            result = json.loads(response.read().decode("utf-8"))
    except HTTPError as error:
        details = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"OpenAI API returned HTTP {error.code}: {details}"
        ) from error
    except URLError as error:
        raise RuntimeError(
            f"Could not reach the OpenAI API: {error.reason}"
        ) from error
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise RuntimeError("OpenAI API returned an invalid JSON response.") from error

    try:
        return result["choices"][0]["message"]["content"].strip()
    except (KeyError, IndexError, AttributeError) as error:
        raise RuntimeError(
            "OpenAI API response did not contain an assistant reply."
        ) from error

## Start chatting

The `messages` list stores both user questions and assistant answers, allowing the model to respond to follow-ups in context. Each new chat starts with a fresh history.

In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Set it in the environment before starting Jupyter."
    )

model = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Lab | M-Shots FAQ chatbot. Type 'quit' or 'exit' to end.")
while True:
    try:
        question = input("\nYou: ").strip()
    except EOFError:
        print()
        break

    if question.lower() in {"quit", "exit"}:
        break
    if not question:
        continue

    messages.append({"role": "user", "content": question})
    try:
        answer = ask_openai(messages, api_key, model)
    except RuntimeError as error:
        messages.pop()
        print(f"Error: {error}")
        continue

    messages.append({"role": "assistant", "content": answer})
    print(f"Bot: {answer}")

Lab | M-Shots FAQ chatbot. Type 'quit' or 'exit' to end.
Bot: Hello! How can I assist you today with the Lab | M-Shots exercise?
Bot: To complete the lab, follow these steps:

1. Open the Jupyter notebook provided for the lab.
2. Carefully read the instructions given in the notebook.
3. Answer each exercise as you progress through the notebook.
4. Test your answers in the provided cells to make sure they work as expected.

If you have any specific questions about the exercises or need help with a particular part, feel free to ask!
Bot: I'm sorry, but I can only provide information related to the Lab | M-Shots exercise. The capital of France is Paris, but if you have questions about the lab, feel free to ask!
Bot: The FAQ does not specifically mention failing the lab. However, to successfully complete it, you should follow the instructions in the Jupyter notebook, answer each exercise correctly, and submit your work as required. If you have concerns about your understanding of the mater